# Week 6, Lab 2 — Build / extend a local MCP server


In [ ]:
WEEK = 'Week 6'
LAB = 'Lab 2 — build server'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


In [ ]:
print((ROOT / "6_mcp" / "servers" / "local_tools_server.py").read_text()[:2000])


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import textwrap

src = textwrap.dedent('''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("demo")

@mcp.tool()
def shout(text: str) -> str:
    """Uppercase the text."""
    return text.upper()

if __name__ == "__main__":
    mcp.run()
''')
path = ROOT / "6_mcp" / "servers" / "_demo_shout.py"
path.write_text(src, encoding="utf-8")

async def demo():
    params = StdioServerParameters(command="python", args=[str(path)])
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            print(await s.list_tools())
            print(await s.call_tool("shout", {"text": "mcp is a protocol"}))

await demo()


Add your own course topics to the KB in `local_tools_server.py`.
